# Barra模型因子归因 — 研报复现

## 研报信息
- **标题**: Barra模型基于持仓数据从因子角度对收益进行拆解
- **来源**: 华泰证券研究所 (2020年8月)
- **核心**: 利用Barra模型将组合收益分解到各公共因子，分析收益来源分布

## Barra模型三步流程
1. **Step 1** — 计算公共因子暴露矩阵X（标准化）
   $$\beta_{ij} = \frac{x_{ij} - \bar{x}_j}{\mathrm{std}(x_j)}$$
2. **Step 2** — 计算公共因子收益率矩阵F（横截面回归）
   $$R_i = \sum_j \beta_{ij} F_j + \varepsilon_i$$
3. **Step 3** — 计算基金因子暴露b（时间序列回归）
   $$R_{pt} = \sum_j F_{jt} b_j + \varepsilon_t$$

## 0. 环境配置

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
src_dir = os.path.join(project_root, 'source')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print(f'pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}')

In [ ]:
from source.data_loader import BarraDataLoader
from source.factor import (
    BarraFactorAttribution, FactorExposureCalculator,
    FactorReturnCalculator, FundExposureCalculator, FactorAttributionResult
)
from source.backtest import (
    BarraRollingBacktest, BacktestConfig,
    AttributionStabilityTest, PerformanceSummary
)
from source.plot import BarraVisualizer
print('模块导入完成')

## 1. 数据获取

In [ ]:
# 参数配置
FUND_CODE    = '019888'
START_DATE   = '2022-01-01'
END_DATE     = '2024-12-31'
FREQ         = 'monthly'
ROLL_WINDOW  = 24
RF_RATE      = 0.03

print(f'标的: {FUND_CODE}  期间: {START_DATE}~{END_DATE}  频率: {FREQ}')

In [ ]:
loader = BarraDataLoader()

# 基金收益率
fund_returns = loader.get_fund_returns(FUND_CODE, START_DATE, END_DATE, FREQ)
if len(fund_returns) == 0:
    print('使用模拟基金收益率')
    np.random.seed(42)
    dates = pd.date_range(START_DATE, END_DATE, freq='M')
    fund_returns = pd.Series(
        np.random.randn(len(dates)) * 0.15 / np.sqrt(12) + 0.08 / 12,
        index=dates
    )

print(f'基金收益率: {len(fund_returns)} 条, 年化 {((1+fund_returns).prod()**(12/len(fund_returns))-1)*100:.2f}%')

In [ ]:
# 因子收益率矩阵
factor_returns = loader.get_factor_returns(START_DATE, END_DATE, FREQ)
print(f'因子收益率矩阵: {factor_returns.shape}')
factor_returns.describe().T.round(4)

## 2. Step 1 — 因子暴露矩阵X
$$\beta_{ij} = \frac{x_{ij} - \bar{x}_j}{\mathrm{std}(x_j)}$$

In [ ]:
# 持仓 & 因子暴露
holdings = loader.get_fund_holdings(FUND_CODE, START_DATE, END_DATE)

if len(holdings) > 0 and 'stock_code' in holdings.columns:
    codes = holdings['stock_code'].unique().tolist()[:50]
    factor_exposure = loader.get_stock_factor_exposure(codes)
else:
    np.random.seed(42)
    n_stocks = 50
    fnames = ['SIZE','BOOK_TO_PRICE','MOMENTUM','VOLATILITY','QUALITY','GROWTH','LEVERAGE','LIQUIDITY']
    factor_exposure = pd.DataFrame(
        np.random.randn(n_stocks, len(fnames)),
        columns=fnames,
        index=[f'{600000+i}' for i in range(n_stocks)]
    )
    factor_exposure['SIZE'] = np.random.uniform(18, 25, n_stocks)

standardized = FactorExposureCalculator.standardize_exposure(factor_exposure)
print(f'标准化因子暴露: {standardized.shape}')
standardized.head()

## 3. Step 2 — 因子收益率矩阵F（横截面回归）
$$R_i = \sum_j \beta_{ij} F_j + \varepsilon_i$$

In [ ]:
# 因子收益率时序图
fig, axes = plt.subplots(len(factor_returns.columns), 1,
                         figsize=(14, 2.5*len(factor_returns.columns)), sharex=True)
for i, col in enumerate(factor_returns.columns):
    ax = axes[i]
    d = factor_returns[col]*100
    c = ['#2ecc71' if v>=0 else '#e74c3c' for v in d]
    ax.bar(d.index, d.values, color=c, alpha=0.7, width=20)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_ylabel(f'{col}(%)', fontsize=9)
    ax.grid(True, alpha=0.2, axis='y')
axes[-1].set_xlabel('日期')
fig.suptitle('Barra因子收益率时序', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Step 3 — 基金因子暴露b（时间序列回归）
$$R_{pt} = \sum_j F_{jt} b_j + \varepsilon_t$$

In [ ]:
attribution = BarraFactorAttribution()
result = attribution.run(
    fund_returns, factor_returns,
    fund_code=FUND_CODE,
    period=f'{START_DATE}~{END_DATE}'
)

In [ ]:
# 因子暴露详表
tbl = pd.DataFrame({
    'factor':   result.factor_names,
    'b':        result.b,
    't_stat':   result.t_stats,
    'p_value':  result.p_values,
    'sig':      ['***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '' for p in result.p_values],
    'contrib%': [result.factor_contributions.get(n,0)*100 for n in result.factor_names]
})
print(tbl.to_string(index=False))
print(f'\nAlpha(年化): {((1+result.alpha)**12-1)*100:.2f}%  R2: {result.r_squared:.4f}')

## 5. 滚动回归回测

In [ ]:
config = BacktestConfig(rolling_window=ROLL_WINDOW, min_periods=12,
                        start_date=START_DATE, end_date=END_DATE)
backtest = BarraRollingBacktest(config)
bt = backtest.run(fund_returns, factor_returns, fund_code=FUND_CODE)

In [ ]:
if bt:
    rolling_exp  = bt['rolling_exposure']
    rolling_sig  = bt['rolling_significance']
    rolling_rsq  = bt['rolling_r_squared']
    rolling_alpha= bt['rolling_alpha']
    print(rolling_exp.describe().round(4))

## 6. 归因稳定性检验

In [ ]:
if bt:
    acf = AttributionStabilityTest.test_exposure_autocorrelation(rolling_exp, lags=3)
    print('因子暴露自相关性:'); print(acf.round(4))
    persist = AttributionStabilityTest.test_significance_persistence(rolling_sig)
    print('\n显著性持续性:'); print(persist)

## 7. 绩效指标汇总

In [ ]:
metrics = PerformanceSummary.calculate(fund_returns, factor_returns, result)
PerformanceSummary.print_summary(metrics, fund_code=FUND_CODE)

## 8. 可视化

In [ ]:
viz = BarraVisualizer(output_dir=os.path.join(project_root, 'output'))

In [ ]:
viz.plot_factor_exposure(result, title=f'Barra因子暴露 | {FUND_CODE}')
plt.show()

In [ ]:
viz.plot_factor_contribution_waterfall(result, title=f'Barra因子贡献分解 | {FUND_CODE}')
plt.show()

In [ ]:
if bt:
    viz.plot_rolling_exposure(rolling_exp, rolling_sig,
                              title=f'滚动因子暴露 | {FUND_CODE} ({ROLL_WINDOW}月)')
    plt.show()

In [ ]:
if bt:
    viz.plot_significance_heatmap(rolling_sig, title=f'因子显著性 | {FUND_CODE}')
    plt.show()

In [ ]:
if bt:
    viz.plot_alpha_time_series(rolling_alpha, title=f'滚动Alpha | {FUND_CODE}')
    plt.show()

In [ ]:
if bt:
    viz.plot_attribution_dashboard(result, rolling_exp, rolling_alpha,
                                    fund_code=FUND_CODE)
    plt.show()

## 9. 结果导出

In [ ]:
out_dir = os.path.join(project_root, 'output')
os.makedirs(out_dir, exist_ok=True)

# 因子暴露CSV
exp_out = pd.DataFrame({
    'factor': result.factor_names, 'exposure_b': result.b,
    't_stat': result.t_stats, 'p_value': result.p_values,
    'sig': ['***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '' for p in result.p_values],
    'contrib_pct': [result.factor_contributions.get(n,0)*100 for n in result.factor_names]
})
exp_out.to_csv(os.path.join(out_dir, f'{FUND_CODE}_barra_factor_exposure.csv'),
               index=False, encoding='utf-8-sig')

# 滚动暴露
if bt:
    rolling_exp.to_csv(os.path.join(out_dir, f'{FUND_CODE}_rolling_exposure.csv'),
                       encoding='utf-8-sig')
    rolling_alpha.to_csv(os.path.join(out_dir, f'{FUND_CODE}_rolling_alpha.csv'),
                         encoding='utf-8-sig')

factor_returns.to_csv(os.path.join(out_dir, 'factor_returns.csv'), encoding='utf-8-sig')

print('已导出:')
for f in os.listdir(out_dir):
    print(f'  {f}  ({os.path.getsize(os.path.join(out_dir,f))/1024:.1f} KB)')

In [ ]:
# 文字报告
report_lines = []
report_lines.append('='*70)
report_lines.append(f'Barra因子归因分析报告  基金: {FUND_CODE}')
report_lines.append('='*70)
report_lines.append(f'期间: {START_DATE}~{END_DATE}  频率: {FREQ}')
report_lines.append('')
report_lines.append('[因子暴露结果]')
for i, name in enumerate(result.factor_names):
    if i < len(result.b):
        sig = '***' if result.p_values[i]<.001 else '**' if result.p_values[i]<.01 else '*' if result.p_values[i]<.05 else ''
        report_lines.append(f'  {name:20s}  b={result.b[i]:>8.4f}  p={result.p_values[i]:.4f} {sig}')
report_lines.append('')
report_lines.append(f'Alpha(年化): {((1+result.alpha)**12-1)*100:.2f}%')
report_lines.append(f'R2: {result.r_squared:.4f}  adj-R2: {result.adj_r_squared:.4f}')
report_lines.append('')
report_lines.append('[绩效指标]')
report_lines.append(f'  年化收益: {metrics["annual_return"]*100:.2f}%')
report_lines.append(f'  年化波动: {metrics["annual_volatility"]*100:.2f}%')
report_lines.append(f'  夏普比率: {metrics["sharpe_ratio"]:.4f}')
report_lines.append(f'  最大回撤: {metrics["max_drawdown"]*100:.2f}%')

report_text = '\n'.join(report_lines)
with open(os.path.join(out_dir, f'{FUND_CODE}_barra_report.txt'), 'w', encoding='utf-8') as f:
    f.write(report_text)

print(report_text)